# Day 3 — Phase 1: Confirm the Retrieval Handoff (Colab-ready)

This notebook implements **Phase 1** of the Day 3 implementation plan:
1. Load the NHLBI 2014 *Evidence-Based Management of Sickle Cell Disease* PDF.
2. Chunk the pages.
3. Build the Chroma vector index.
4. Run a retrieval query and validate that every returned chunk has the citation metadata (`chunk_id`, `page_number`, `citation`, `title`, `document_id`) needed for grounded generation.

It is designed to run in **Google Colab** (it will clone the repo, install requirements, and load `GROQ_API_KEY` from a Colab secret) **or** locally (it will use the parent directory as the repo root).


In [1]:
import os
import sys
import subprocess
import pathlib

# Detect Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Cloning repo and installing requirements...")
    repo_path = pathlib.Path("/content/day3-work")
    if not repo_path.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Abdelrhamanhaitham22/day3-work.git", str(repo_path)],
            check=True,
        )
    os.chdir(repo_path)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
    sys.path.insert(0, str(repo_path))

    # Load GROQ_API_KEY from Colab secret if available
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        print("GROQ_API_KEY loaded from Colab secret")
    except Exception:
        print("No GROQ_API_KEY Colab secret found. Set it later if you want LLM generation.")
else:
    repo_root = pathlib.Path("..").resolve()
    sys.path.insert(0, str(repo_root))
    print("Running locally. Repo root:", repo_root)


Running locally. Repo root: /home/ubuntu/repos/day3-work


In [2]:
import config
import ingest
import query


### Step 1: Load the PDF pages

In [3]:
pages = ingest.load_pdfs()
print(f"Loaded {len(pages)} pages from {config.PDF_PATH}")


Loaded 105 pages from /home/ubuntu/repos/day3-work/references/56-364NFULL.pdf


### Step 2: Split pages into citation-ready chunks

In [4]:
chunks = ingest.chunk_documents(
    pages,
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP,
)
print(f"Created {len(chunks)} chunks")
assert len(chunks) > 0, "No chunks were produced"

sample = chunks[0]
print("Sample chunk_id:", sample.metadata["chunk_id"])
print("Sample page_number:", sample.metadata["page_number"])


Created 510 chunks
Sample chunk_id: nhlbi-scd-2014-CH-0001
Sample page_number: 1


### Step 3: Build the Chroma vector index

In [5]:
vectordb = query.build_index(chunks)
print("Vector index built and ready")


/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector index built and ready


### Step 4: Retrieve evidence for a sample SCD question

In [6]:
test_question = "When should hydroxyurea therapy be started in adults with sickle cell anemia?"
results = query.retrieve(vectordb, test_question, k=config.TOP_K)

print(f"Top {len(results)} retrieved chunks for: {test_question}\n")
for doc, score in results:
    meta = doc.metadata
    print(f"chunk_id={meta['chunk_id']} | page={meta['page_number']} | score={score:.4f}")
    print(f"citation: {meta['citation']}")
    print(doc.page_content[:300].replace("\n", " "))
    print("---")


Top 4 retrieved chunks for: When should hydroxyurea therapy be started in adults with sickle cell anemia?

chunk_id=nhlbi-scd-2014-CH-0397 | page=77 | score=0.8635
citation: National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month  period, treat with hydroxyurea.  (Strong Recommendation, High-Quality Evidence)  3. In adults with SCA who have sickle cell-associated pain that interferes with daily activities and qual
---
chunk_id=nhlbi-scd-2014-CH-0361 | page=71 | score=0.8424
citation: National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.
Chapter 5: Hydroxyurea Therapy in the Management of Sickle Cell Disease  Introduction  This chapter addresses the use of hydroxyurea (also called hydroxycarbamide) in adults and children who have  SCD

In [7]:
required_fields = {"chunk_id", "page_number", "citation", "title", "document_id"}
for doc, score in results:
    missing = required_fields - set(doc.metadata.keys())
    assert not missing, f"Missing metadata fields: {missing}"
print("Phase 1 validation PASSED: every retrieved chunk has the required citation metadata.")


Phase 1 validation PASSED: every retrieved chunk has the required citation metadata.


### Optional: confirm the Groq API key is usable

This cell does a minimal Groq API call. It is **optional** for Phase 1 because Phase 1 only validates retrieval, but it proves the key you saved in Colab is ready for Phase 2/3 generation.


In [8]:
if os.environ.get("GROQ_API_KEY"):
    try:
        from groq import Groq
        client = Groq(api_key=os.environ["GROQ_API_KEY"])
        resp = client.chat.completions.create(
            model=config.GROQ_MODEL,
            messages=[{"role": "user", "content": "Say hello."}],
            temperature=0,
            max_tokens=50,
        )
        print("Groq key works. Model reply:", resp.choices[0].message.content.strip())
    except Exception as e:
        print("Groq key check failed (optional):", e)
else:
    print("GROQ_API_KEY not set; skipping optional key check.")


Groq key works. Model reply: Hello! 👋 How can I assist you today?


## Phase 1 complete

The evidence layer is ready:
- PDF loaded and chunked
- Vector index built
- Retrieval returns chunks with `chunk_id`, `page_number`, `citation`, `title`, and `document_id`

Next: **Phase 2** — select `top_k`, `chunk_size`, and `chunk_overlap` using the Day 1/2 experiments, then move to the grounding rules and structured answer format in **Phase 4**.
